In [1]:
pip install -U deepeval

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 12.5 MB/s  0:00:00
  Attempting uninstall: deepeval
    Found existing installation: deepeval 4.3.2
    Uninstalling deepeval-4.3.2:
      Successfully uninstalled deepeval-4.3.2
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1) Load environment variables securely

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load local developer secrets without committing them to the repo.
# In Jenkins or other CI, these values will come from credentials or env variables.
env_path = Path(r"D:/Udemy_AI_project/Session1_Intro/.env")
load_dotenv(env_path, override=False)

openai_key = os.getenv("OPENAI_API_KEY")
confident_key = os.getenv("CONFIDENT_API_KEY")

print("OpenAI key loaded:", bool(openai_key))
print("Confident key loaded:", bool(confident_key))

if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key

if confident_key:
    os.environ["CONFIDENT_API_KEY"] = confident_key

OpenAI key loaded: True
Confident key loaded: True


### 2) OpenAI + DeepEval evaluation

#### Metric Type: Answer Relevency Metric

In [3]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"D:\Udemy_AI_project\Session1_Intro\.env")

from deepeval.models import OpenAIModel
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric

openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise RuntimeError("OPENAI_API_KEY is missing. Set it in the environment or .env file.")

llm_model = OpenAIModel(model="gpt-4o-mini", api_key=openai_key)
metric = AnswerRelevancyMetric(async_mode=False, model=llm_model)

test_case = LLMTestCase(
    input="Who is the current president of United States Of America?",
    actual_output="Joe Biden",
    retrieval_context=["The current president of the United States is Joe Biden."]
)

metric.measure(test_case)
print("Answer Relevancy Score:", metric.score)

Output()

Answer Relevancy Score: 1.0


### 3) Confident AI login and reporting (separate flow)

In [4]:
import deepeval
import os

confident_key = os.getenv("CONFIDENT_API_KEY")
if not confident_key:
    raise RuntimeError("CONFIDENT_API_KEY is missing. Set it in the environment or .env file.")

deepeval.login(confident_key)
print("Confident AI login successful")

🎉🥳 Congratulations! You've successfully logged in! 🙌

Confident AI login successful


#### Metric Type: Contextual precission Metric

In [5]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import ContextualPrecisionMetric

contextual_precision_metric = ContextualPrecisionMetric(async_mode=False, model=llm_model)

test_case = LLMTestCase(
    input = "Who is the current president of USA in 2024?",
    # In future the output should come from LLM or an Agent or RAG
    actual_output = "Donald trump",
    # RAG - Vector DB, AI Agent - Agent Tools, LLM - LLM invoke response
    retrieval_context=[
            "Donald Trump won the 2024 presidential election.",
            "There was a cat in the White House.",
        ],
    expected_output = "Donald Trump was president back in 2024"
)

contextual_precision_metric.measure(test_case=test_case)
print(contextual_precision_metric.score)
print(contextual_precision_metric.success)
print(contextual_precision_metric.score_breakdown)
print(contextual_precision_metric.verdicts)
print(contextual_precision_metric.reason)

Output()

1.0
True
None
[ContextualPrecisionVerdict(verdict=<Verdict.YES: 'yes'>, reason="The statement 'Donald Trump won the 2024 presidential election.' directly answers the question about who the current president is in 2024."), ContextualPrecisionVerdict(verdict=<Verdict.NO: 'no'>, reason="'There was a cat in the White House.' is irrelevant to the question of who the current president is.")]
The score is 1.00 because the first node directly answers the question about the current president in 2024, while the second node, ranked lower, provides irrelevant information about a cat in the White House, which does not pertain to the inquiry.


Evaluate test without using standalone : using 'Evaluate'(Confident AI)

In [6]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"D:\Udemy_AI_project\Session1_Intro\.env")

from deepeval.models import OpenAIModel
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate

openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise RuntimeError("OPENAI_API_KEY is missing. Set it in the environment or .env file.")

llm_model = OpenAIModel(model="gpt-4o-mini", api_key=openai_key)
metric = AnswerRelevancyMetric(async_mode=False, model=llm_model)

test_case = LLMTestCase(
    input="Who is the current president of United States Of America?",
    actual_output="Joe Biden",
    retrieval_context=["The current president of the United States is Joe Biden."]
)

evaluate(test_cases=[test_case], metrics=[metric])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=11851605;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=11851608;https://cloud.confident-ai.com/project/cmu80ye8f002fo30tefmo2u6f/test-runs/cmwb0gvp70007f8012feb377v\https://cloud.confident-ai.com/project/cmu80ye8f002fo30tefmo2u6f/test-runs/cmwb0gvp70007f8012feb377v]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the response directly addresses the question about the current president of the United States without any irrelevant statements.', strict_mode=False, flaky=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.00020145000000000002, input_tokens=1055, output_tokens=72, verbose_logs='Statements:\n[\n    "Joe Biden"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]', expectations_data=None)], conversational=False, index=0, multimodal=False, input='Who is the current president of United States Of America?', actual_output='Joe Biden', expected_output=None, context=None, retrieval_context=['The current president of the United States is Joe Biden.'], turns=None, metadata=None, classifications=None, expectations=None)], confident_link='https://

Multiple test cases : Evaluate (Confident AI)

In [7]:
import os
import threading
import asyncio
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"D:\Udemy_AI_project\Session1_Intro\.env")

from deepeval.models import OpenAIModel
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate
from deepeval.evaluate.configs import AsyncConfig

openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise RuntimeError("OPENAI_API_KEY is missing. Set it in the environment or .env file.")

confident_key = os.getenv("CONFIDENT_API_KEY")
if not confident_key:
    raise RuntimeError("CONFIDENT_API_KEY is missing. Set it in the environment or .env file.")

llm_model = OpenAIModel(model="gpt-4o-mini", api_key=openai_key)
metric = AnswerRelevancyMetric(async_mode=True, model=llm_model)

test_case1 = LLMTestCase(
    input="Who is the current president of United States Of America?",
    actual_output="Joe Biden",
    expected_output="Joe Biden",
    retrieval_context=["The current president of the United States is Joe Biden."]
)

test_case2 = LLMTestCase(
    input="Who built the Taj Mahal?",
    actual_output="Aurangzeb Alamgir",
    expected_output="Shah Jahan",
    retrieval_context=["The Taj Mahal was built by Mughal Emperor Shah Jahan."]
)

print("Evaluating test cases...")


# Python 3.14 + Jupyter workaround:
# deepeval's async path runs `loop.run_until_complete()` internally, which breaks
# under Jupyter's already-running (and nest_asyncio-patched) event loop with
# "RuntimeError: Timeout should be used inside a task".
# Forcing run_async=False uses the fully synchronous execution path, which avoids
# the nested loop entirely. We also run it on a fresh-thread loop for safety.

# def run_evaluation():
#     loop = asyncio.new_event_loop()
#     asyncio.set_event_loop(loop)
#     try:
#         return evaluate(
#             test_cases=[test_case1, test_case2],
#             metrics=[metric],
#             async_config=AsyncConfig(run_async=False),
#         )
#     finally:
#         loop.close()


# results = run_evaluation()
# print("Results:", results)

evaluate(test_cases=[test_case1, test_case2], metrics=[metric], async_config=AsyncConfig(run_async=False))

Evaluating test cases...


✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              Who built the Taj Mahal?                                                             │
│  │     Actual Output:      Aurangzeb Alamgir                                                                    │
│  │     Expected Output:    Shah Jahan                                                                           │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Relevancy │ 0.00  │ 0.50      │ The score is 0.00 because the response incorrectly        │
│              │                  │       │           │ attributes the construction of the Taj Mahal to           │
│              │                  │       │           │ Aurangzeb Alamgir instead of Shah Jahan, which is a       │
│              │                  │       │           │ fundamental error in addressing the question.             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score          ┃ Pass Rate                                    ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 0.50                   │ 50.00% | passed=1 | failed=1                 │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=11851610;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=11851613;https://cloud.confident-ai.com/project/cmu80ye8f002fo30tefmo2u6f/test-runs/cmwb0hblk001nf801p75iqbfi\https://cloud.confident-ai.com/project/cmu80ye8f002fo30tefmo2u6f/test-runs/cmwb0hblk001nf801p75iqbfi]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the response directly addresses the question about the current president of the United States without any irrelevant statements.', strict_mode=False, flaky=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.00020145000000000002, input_tokens=1055, output_tokens=72, verbose_logs='Statements:\n[\n    "Joe Biden"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]', expectations_data=None)], conversational=False, index=0, multimodal=False, input='Who is the current president of United States Of America?', actual_output='Joe Biden', expected_output='Joe Biden', context=None, retrieval_context=['The current president of the United States is Joe Biden.'], turns=None, metadata=None, classifications=None, expectations=None), TestResult(name='t

## Evaluate with Golden Dataset and EvaluationDataset

In [8]:
from deepeval.models import OpenAIModel
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate
from deepeval.dataset import EvaluationDataset, Golden 

openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise RuntimeError("OPENAI_API_KEY is missing. Set it in the environment or .env file.")

confident_key = os.getenv("CONFIDENT_API_KEY")
if not confident_key:
    raise RuntimeError("CONFIDENT_API_KEY is missing. Set it in the environment or .env file.")

llm_model = OpenAIModel(model="gpt-4o-mini", api_key=openai_key)
metric = AnswerRelevancyMetric(async_mode=True, model=llm_model)

golden_dataset = Golden(
                input="What is the capital of France?",
                expected_output="The capital of France is Paris.",
                context=["France is a country in Europe. Its capital city is Paris."],
            )
    

dataset = EvaluationDataset()
dataset.add_golden(golden_dataset)

In [9]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(id=None, input='What is the capital of France?', actual_output=None, expected_output='The capital of France is Paris.', context=['France is a country in Europe. Its capital city is Paris.'], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, token_cost=None, input_token_count=None, output_token_count=None, source_file=None, name=None, custom_column_key_values=None, expectations=None, expected_labels=None, multimodal=False, images_mapping=None)], _alias=None, _id=None, _version=None, _multi_turn=False)

### Create Test case from Golden (From the above dataset)

In [10]:
for golden_dataset in dataset.goldens:
    test_case = LLMTestCase(
        input=golden_dataset.input,
        expected_output=golden_dataset.expected_output,
        actual_output="Paris",
        retrieval_context=golden_dataset.context
    )
    dataset.add_test_case(test_case)

evaluate(test_cases=dataset.test_cases, metrics=[metric], async_config=AsyncConfig(run_async=False))    

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 1.00                  │ 100.00% | passed=1 | failed=0                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=11851615;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=11851618;https://cloud.confident-ai.com/project/cmu80ye8f002fo30tefmo2u6f/test-runs/cmwb0ho66001fg601367wbfqh\https://cloud.confident-ai.com/project/cmu80ye8f002fo30tefmo2u6f/test-runs/cmwb0ho66001fg601367wbfqh]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because the response directly answered the question about the capital of France without any irrelevant statements.', strict_mode=False, flaky=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.00019754999999999998, input_tokens=1045, output_tokens=68, verbose_logs='Statements:\n[\n    "Paris"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]', expectations_data=None)], conversational=False, index=0, multimodal=False, input='What is the capital of France?', actual_output='Paris', expected_output='The capital of France is Paris.', context=None, retrieval_context=['France is a country in Europe. Its capital city is Paris.'], turns=None, metadata=None, classifications=None, expectations=None)], confident_link='https://cloud.confident-ai.com/proj

### Creating Evaluation Dataset as Golden in confident AI

##### Creating Data

In [11]:
test_data = [
    {
        "input": "What is the capital of France?",
        "expected_output": "The capital of France is Paris.",
    },
    {
        "input": "Who wrote 'Pride and Prejudice'?",
        "expected_output": "Jane Austen wrote 'Pride and Prejudice'."
    }
]

In [12]:
from deepeval.models import OpenAIModel
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate
from deepeval.dataset import EvaluationDataset, Golden 

goldens_dataset = []

for data in test_data:
    golden_dataset = Golden(
        input=data["input"],
        expected_output=data["expected_output"],
        context=[f"Context for: {data['input']}"]
    )
    goldens_dataset.append(golden_dataset)

new_ConfidentAI_Dataset = EvaluationDataset(goldens=goldens_dataset)
new_ConfidentAI_Dataset 

EvaluationDataset(test_cases=[], goldens=[Golden(id=None, input='What is the capital of France?', actual_output=None, expected_output='The capital of France is Paris.', context=['Context for: What is the capital of France?'], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, token_cost=None, input_token_count=None, output_token_count=None, source_file=None, name=None, custom_column_key_values=None, expectations=None, expected_labels=None, multimodal=False, images_mapping=None), Golden(id=None, input="Who wrote 'Pride and Prejudice'?", actual_output=None, expected_output="Jane Austen wrote 'Pride and Prejudice'.", context=["Context for: Who wrote 'Pride and Prejudice'?"], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, token_cost=None, input_token_count=None, output_token_count=None, source_file=None, name=None, custom_column_key_values=None, expectations=None, expected_labels

### Push the Dataset to Confident AI

In [16]:
new_ConfidentAI_Dataset.push(alias="TestGoldenDataset")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=11851623;https://cloud.confident-ai.com/project/cmu80ye8f002fo30tefmo2u6f/datasets/cmua1lttt000vnx0t70b23j72\https://cloud.confident-ai.com/project/cmu80ye8f002fo30tefmo2u6f/datasets/cmua1lttt000vnx0t70b23j72]8;;\

#### Pull the Dataset from Confident AI 

In [19]:
from deepeval.dataset import EvaluationDataset

clouddataset = EvaluationDataset()
clouddataset.pull(alias="TestGoldenDataset", finalized=False)
clouddataset

Output()

ConfidentApiError: An active trial or paid plan is required. Start a trial or upgrade in your Organization billing settings to continue using this dataset.

### Now Prepare our Complete flow of Test Cases to Evaluate Our Records

In [20]:
def mock_llm_response(input_text):
    if "capital of France" in input_text:
        return "The capital of France is Paris."
    elif "Pride and Prejudice" in input_text:
        return "Jane Austen wrote 'Pride and Prejudice'."
    else:
        return "I don't know."

In [21]:
from deepeval.test_case import LLMTestCase

# Reset so re-running this cell doesn't append duplicate test cases
clouddataset.test_cases = []
count = 1
for golden_dataset in clouddataset.goldens:
    test_case = LLMTestCase(
        input=golden_dataset.input,
        expected_output=golden_dataset.expected_output,
        actual_output=mock_llm_response(golden_dataset.input),
        retrieval_context=golden_dataset.context
    )
    clouddataset.add_test_case(test_case)
    print(f"Test case {count} added for input: {golden_dataset.input}")
    count += 1

### Prepare Test Case to Evaluate Our Records With the Dataset Present in Confident AI 

In [22]:
import os
from dotenv import load_dotenv
from deepeval.evaluate import evaluate
from deepeval.evaluate.configs import AsyncConfig
from deepeval.models import OpenAIModel
from deepeval.metrics import AnswerRelevancyMetric

load_dotenv(r"D:\Udemy_AI_project\Session1_Intro\.env")
openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise RuntimeError("OPENAI_API_KEY is missing. Set it in the environment or .env file.")

llm_model = OpenAIModel(model="gpt-4o-mini", api_key=openai_key)
metric = AnswerRelevancyMetric(async_mode=True, model=llm_model)

evaluate(test_cases=clouddataset.test_cases, metrics=[metric], async_config=AsyncConfig(run_async=False))

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

No test cases found, please try again.

EvaluationResult(test_results=[], confident_link=None, test_run_id=None)

## Using Local LLM For Evaluation

##### Command to start the local llm Ollama

In [23]:
!deepeval set-ollama --model llama3.2:latest

🙌 Congratulations! You're now using a local Ollama model `llama3.2:latest` for
all evals that require an LLM.


In [24]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=r"D:\Udemy_AI_project\Session1_Intro\.env")

from deepeval.models import OllamaModel
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.evaluate import evaluate
from deepeval.evaluate.configs import AsyncConfig

#Now because we have set the ollama model, we can use it for evaluation 
#And we dont need to use the OpenAI API key anymore, we can use the ollama model for evaluation

confident_key = os.getenv("CONFIDENT_API_KEY")
if not confident_key:
    raise RuntimeError("CONFIDENT_API_KEY is missing. Set it in the environment or .env file.")

llm_model = OllamaModel(model="llama3.2:latest", base_url="http://localhost:11434")
metric = AnswerRelevancyMetric(async_mode=True, model=llm_model)

test_case1 = LLMTestCase(
    input="Who is the current president of United States Of America?",
    actual_output="Joe Biden",
    expected_output="Joe Biden",
    retrieval_context=["The current president of the United States is Joe Biden."]
)

test_case2 = LLMTestCase(
    input="Who built the Taj Mahal?",
    actual_output="Aurangzeb Alamgir",
    expected_output="Shah Jahan",
    retrieval_context=["The Taj Mahal was built by Mughal Emperor Shah Jahan."]
)

print("Evaluating test cases...")
evaluate(test_cases=[test_case1, test_case2], metrics=[metric], async_config=AsyncConfig(run_async=False))

Evaluating test cases...


✨ You're running DeepEval's latest Answer Relevancy Metric! (using llama3.2:latest (Ollama), strict=False, 
async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              Who built the Taj Mahal?                                                             │
│  │     Actual Output:      Aurangzeb Alamgir                                                                    │
│  │     Expected Output:    Shah Jahan                                                                           │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Relevancy │ 0.00  │ 0.50      │ The score is 0.00 because there are no irrelevant         │
│              │                  │       │           │ statements in the actual output, making it a perfect      │
│              │                  │       │           │ score.                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score          ┃ Pass Rate                                    ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 0.50                   │ 50.00% | passed=1 | failed=1                 │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=11851625;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=11851628;https://cloud.confident-ai.com/project/cmu80ye8f002fo30tefmo2u6f/test-runs/cmwb0zfrc006gg7015k8xvfai\https://cloud.confident-ai.com/project/cmu80ye8f002fo30tefmo2u6f/test-runs/cmwb0zfrc006gg7015k8xvfai]8;;\

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=1.0, reason='The score is 1.00 because there are no irrelevant statements in the actual output, making it a perfect answer that directly addresses the question.', strict_mode=False, flaky=False, evaluation_model='llama3.2:latest (Ollama)', error=None, evaluation_cost=0.0, input_tokens=0, output_tokens=0, verbose_logs='Statements:\n[\n    "Joe Biden"\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]', expectations_data=None)], conversational=False, index=0, multimodal=False, input='Who is the current president of United States Of America?', actual_output='Joe Biden', expected_output='Joe Biden', context=None, retrieval_context=['The current president of the United States is Joe Biden.'], turns=None, metadata=None, classifications=None, expectations=None), TestResult(name='test_case_1', succ